# 국가별 법규 PDF 수집 실습

공식 기관의 진입 페이지를 기준으로 PDF 링크를 찾고, 국가/기관 폴더 아래에 파일을 저장하는 노트북이다. 먼저 수집 흐름이 안정적으로 도는지 확인하는 목적이다.


In [ ]:
# %pip install pandas requests python-dotenv

In [ ]:
from pathlib import Path
from urllib.parse import urljoin, urlparse
import re
import time
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

RAG_ROOT = Path.cwd()
PROJECT_ROOT = RAG_ROOT.parent
DOCUMENT_ROOT = RAG_ROOT / "documents" / "trade_regulations"
DOCUMENT_ROOT.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/136.0 Safari/537.36"
})

retry = Retry(
    total=3,
    connect=3,
    read=3,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
adapter = HTTPAdapter(max_retries=retry)
session.mount("http://", adapter)
session.mount("https://", adapter)

## 1. 수집할 공식 소스 목록을 만든다

메인 페이지 1개만 쓰지 않고, PDF가 있을 가능성이 높은 문서 허브형 seed를 여러 개 등록한다.


In [ ]:
source_rows = [
    {
        "source_id": "kr_customs_forms",
        "country_code": "KR",
        "agency_code": "CUSTOMS",
        "agency_name": "Korea Customs Service",
        "entry_url": "https://www.customs.go.kr/english/cm/cntnts/cntntsView.do?cntntsId=2728&mi=8048",
        "seed_name": "customs_declaration_forms",
        "active": True,
    },
    {
        "source_id": "kr_customs_import",
        "country_code": "KR",
        "agency_code": "CUSTOMS",
        "agency_name": "Korea Customs Service",
        "entry_url": "https://www.customs.go.kr/kcs_english/cm/cntnts/cntntsView.do?cntntsId=2731&mi=8055",
        "seed_name": "import_clearance_guide",
        "active": True,
    },
    {
        "source_id": "kr_customs_export_form",
        "country_code": "KR",
        "agency_code": "CUSTOMS",
        "agency_name": "Korea Customs Service",
        "entry_url": "https://www.customs.go.kr/download/Forms-export.pdf",
        "seed_name": "export_declaration_pdf",
        "active": True,
    },
    {
        "source_id": "kr_customs_import_form",
        "country_code": "KR",
        "agency_code": "CUSTOMS",
        "agency_name": "Korea Customs Service",
        "entry_url": "https://www.customs.go.kr/download/Forms-import.pdf",
        "seed_name": "import_declaration_pdf",
        "active": True,
    },
    {
        "source_id": "kr_customs_simple_form",
        "country_code": "KR",
        "agency_code": "CUSTOMS",
        "agency_name": "Korea Customs Service",
        "entry_url": "https://www.customs.go.kr/download/Customs%20Declaration%20Form.pdf",
        "seed_name": "simplified_customs_form_pdf",
        "active": True,
    },
    {
        "source_id": "kr_mfds_import_food",
        "country_code": "KR",
        "agency_code": "MFDS",
        "agency_name": "Ministry of Food and Drug Safety",
        "entry_url": "https://www.mfds.go.kr/eng/wpge/m_11/de011002l001.do",
        "seed_name": "imported_food_safety",
        "active": True,
    },
    {
        "source_id": "us_cbp_basic",
        "country_code": "US",
        "agency_code": "CBP",
        "agency_name": "U.S. Customs and Border Protection",
        "entry_url": "https://www.cbp.gov/trade/basic-import-export",
        "seed_name": "basic_import_export",
        "active": True,
    },
    {
        "source_id": "us_fda_import_food",
        "country_code": "US",
        "agency_code": "FDA",
        "agency_name": "U.S. Food and Drug Administration",
        "entry_url": "https://www.fda.gov/importing-food-products-united-states",
        "seed_name": "importing_food_products",
        "active": True,
    },
    {
        "source_id": "us_fda_human_food",
        "country_code": "US",
        "agency_code": "FDA",
        "agency_name": "U.S. Food and Drug Administration",
        "entry_url": "https://www.fda.gov/industry/importing-fda-regulated-products/importing-human-foods",
        "seed_name": "importing_human_foods",
        "active": True,
    },
    {
        "source_id": "us_aphis_plant_imports",
        "country_code": "US",
        "agency_code": "APHIS",
        "agency_name": "USDA APHIS",
        "entry_url": "https://www.aphis.usda.gov/plant-imports/how-to-import",
        "seed_name": "plant_imports",
        "active": True,
    },
    {
        "source_id": "us_aphis_animal_product_import",
        "country_code": "US",
        "agency_code": "APHIS",
        "agency_name": "USDA APHIS",
        "entry_url": "https://www.aphis.usda.gov/animal-product-import",
        "seed_name": "animal_product_import",
        "active": True,
    },
    {
        "source_id": "jp_customs_index",
        "country_code": "JP",
        "agency_code": "CUSTOMS",
        "agency_name": "Japan Customs",
        "entry_url": "https://www.customs.go.jp/english/exp-imp/index.htm",
        "seed_name": "exp_imp_index",
        "active": True,
    },
    {
        "source_id": "jp_customs_import",
        "country_code": "JP",
        "agency_code": "CUSTOMS",
        "agency_name": "Japan Customs",
        "entry_url": "https://www.customs.go.jp/english/summary/import.htm",
        "seed_name": "import_summary",
        "active": True,
    },
    {
        "source_id": "jp_customs_export",
        "country_code": "JP",
        "agency_code": "CUSTOMS",
        "agency_name": "Japan Customs",
        "entry_url": "https://www.customs.go.jp/english/summary/export.htm",
        "seed_name": "export_summary",
        "active": True,
    },
    {
        "source_id": "jp_mhlw_imported_foods",
        "country_code": "JP",
        "agency_code": "MHLW",
        "agency_name": "Ministry of Health, Labour and Welfare",
        "entry_url": "https://www.mhlw.go.jp/stf/seisakunitsuite/bunya/kenkou_iryou/shokuhin/yunyu_kanshi/index_00017.html",
        "seed_name": "imported_foods_monitoring",
        "active": True,
    },
    {
        "source_id": "vn_moit_procedures",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/procedures",
        "seed_name": "procedures",
        "active": True,
    },
    {
        "source_id": "vn_moit_legal_goods",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/legal-documents?pages=good",
        "seed_name": "legal_documents_goods",
        "active": True,
    },
    {
        "source_id": "vn_moit_legal_service",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/legal-documents?pages=service",
        "seed_name": "legal_documents_service",
        "active": True,
    },
    {
        "source_id": "vn_moit_legal_agreement",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/legal-documents?pages=agreement",
        "seed_name": "legal_documents_agreement",
        "active": True,
    },
    {
        "source_id": "vn_moit_publications",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/news/post/Publications",
        "seed_name": "publications",
        "active": True,
    },
    {
        "source_id": "vn_moit_use_guide",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/how-to-use-this-site",
        "seed_name": "site_usage_guide",
        "active": True,
    },
    {
        "source_id": "vn_moit_useful_resources",
        "country_code": "VN",
        "agency_code": "MOIT",
        "agency_name": "Vietnam National Trade Repository",
        "entry_url": "https://vntr.moit.gov.vn/useful-resources",
        "seed_name": "useful_resources",
        "active": True,
    },
    {
        "source_id": "eu_trade_markets",
        "country_code": "EU",
        "agency_code": "TRADE",
        "agency_name": "Access2Markets",
        "entry_url": "https://policy.trade.ec.europa.eu/help-exporters-and-importers/accessing-markets_en",
        "seed_name": "accessing_markets",
        "active": True,
    },
    {
        "source_id": "eu_trade_importing",
        "country_code": "EU",
        "agency_code": "TRADE",
        "agency_name": "Access2Markets",
        "entry_url": "https://policy.trade.ec.europa.eu/help-exporters-and-importers/importing-eu_en",
        "seed_name": "importing_eu",
        "active": True,
    },
    {
        "source_id": "eu_trade_leaflet",
        "country_code": "EU",
        "agency_code": "TRADE",
        "agency_name": "Access2Markets",
        "entry_url": "https://trade.ec.europa.eu/access-to-markets/en/assets/leaflet_en.pdf",
        "seed_name": "access2markets_leaflet_pdf",
        "active": True,
    },
    {
        "source_id": "eu_trade_world_markets",
        "country_code": "EU",
        "agency_code": "TRADE",
        "agency_name": "Access2Markets",
        "entry_url": "https://policy.trade.ec.europa.eu/help-exporters-and-importers/eu-companies-accessing-world-markets_en",
        "seed_name": "eu_companies_accessing_world_markets",
        "active": True,
    },
]

source_df = pd.DataFrame(source_rows)
source_df

In [ ]:
source_df.isna().sum()

In [ ]:
source_df.duplicated(subset=["entry_url"]).sum()

## 2. PDF 링크를 찾고 저장 경로를 만든다

페이지 본문 전체를 저장하지 않고, PDF 후보 링크만 골라서 내려받는다. 직접 PDF 링크를 seed로 넣은 경우에는 바로 다운로드 대상으로 본다.


In [ ]:
HREF_RE = re.compile(r'href=["\']([^"\']+)["\']', re.IGNORECASE)
PDF_PATH_KEYWORDS = ["/download/", "/downloadfile/", "/storage/publications/"]
DETAIL_KEYWORDS = [
    "cntntsview.do",
    "selectntt",
    "/summary/",
    "/legal-documents",
    "/procedures",
    "/news/post/",
    "/import",
    "/export",
    "guide",
    "form",
    "manual",
    "permit",
    "inspection",
    "declaration",
]
MAX_DETAIL_LINKS_PER_SEED = 40
PRACTICAL_KEYWORDS = [
    "form",
    "declaration",
    "import",
    "export",
    "clearance",
    "manual",
    "guide",
    "certificate",
    "customs",
]
NOISY_KEYWORDS = [
    "leaflet",
    "interim_results",
    "results_fy",
    "annual",
    "report",
    "strategy",
    "bulletin",
]


def fetch_page(url: str, timeout: int = 20) -> str:
    response = session.get(url, timeout=timeout)
    response.raise_for_status()
    response.encoding = response.encoding or "utf-8"
    return response.text


def is_pdf_candidate(url: str) -> bool:
    lower_url = url.lower()
    lower_path = urlparse(url).path.lower()
    file_name = Path(lower_path).name

    if ".pdf" in lower_url or file_name.endswith(".pdf"):
        return True

    if not file_name:
        return False

    has_pdf_like_name = ".pdf" in file_name or file_name.endswith("pdf")
    has_download_path = any(keyword in lower_path for keyword in PDF_PATH_KEYWORDS)
    return has_download_path and has_pdf_like_name


def is_same_domain(base_url: str, target_url: str) -> bool:
    base_netloc = urlparse(base_url).netloc.lower()
    target_netloc = urlparse(target_url).netloc.lower()
    return base_netloc == target_netloc


def is_detail_candidate(url: str, base_url: str) -> bool:
    lower_url = url.lower()
    parsed = urlparse(url)

    if parsed.scheme not in {"http", "https"}:
        return False
    if not is_same_domain(base_url, url):
        return False
    if is_pdf_candidate(url):
        return False

    path_and_query = f"{parsed.path}?{parsed.query}".lower()
    return any(keyword in path_and_query for keyword in DETAIL_KEYWORDS)


def extract_pdf_links(html: str, base_url: str) -> list[str]:
    candidates = []
    for href in HREF_RE.findall(html or ""):
        absolute_url = urljoin(base_url, href)
        if is_pdf_candidate(absolute_url):
            candidates.append(absolute_url)
    return sorted(set(candidates))


def extract_detail_links(html: str, base_url: str, limit: int = MAX_DETAIL_LINKS_PER_SEED) -> list[str]:
    candidates = []
    seen = set()

    for href in HREF_RE.findall(html or ""):
        absolute_url = urljoin(base_url, href)
        absolute_url = absolute_url.split("#")[0]
        if not absolute_url or absolute_url in seen:
            continue
        if is_detail_candidate(absolute_url, base_url):
            seen.add(absolute_url)
            candidates.append(absolute_url)
        if len(candidates) >= limit:
            break

    return candidates


def normalize_file_name(pdf_url: str, fallback_name: str) -> str:
    parsed = urlparse(pdf_url)
    file_name = Path(parsed.path).name or fallback_name
    file_name = re.sub(r"[^A-Za-z0-9._-]", "_", file_name)
    if not file_name.lower().endswith(".pdf"):
        file_name = f"{file_name}.pdf"
    return file_name


def is_practical_pdf(country_code: str, agency_code: str, file_name: str) -> bool:
    lower_name = file_name.lower()

    if ".hwp.pdf" in lower_name:
        return False

    if any(keyword in lower_name for keyword in NOISY_KEYWORDS):
        return False

    if country_code == "KR" and agency_code == "CUSTOMS":
        return True

    if country_code == "US" and agency_code == "CBP":
        return any(keyword in lower_name for keyword in PRACTICAL_KEYWORDS)

    return False


def build_save_dir(country_code: str, agency_code: str) -> Path:
    save_dir = DOCUMENT_ROOT / country_code / agency_code
    save_dir.mkdir(parents=True, exist_ok=True)
    return save_dir


def download_pdf(pdf_url: str, save_path: Path, timeout: int = 30) -> int:
    response = session.get(pdf_url, timeout=timeout)
    response.raise_for_status()
    save_path.write_bytes(response.content)
    return save_path.stat().st_size


## 3. 링크를 모으고 파일을 저장한다

중복 URL은 한 번만 내려받고, 실패한 seed와 PDF가 없던 seed도 결과 프레임에 남긴다.


In [ ]:
download_rows = []
global_seen_urls = set()

for _, row in source_df[source_df["active"]].iterrows():
    source_id = row["source_id"]
    seed_name = row["seed_name"]
    country_code = row["country_code"]
    agency_code = row["agency_code"]
    entry_url = row["entry_url"]
    save_dir = build_save_dir(country_code, agency_code)
    detail_rows = []

    if is_pdf_candidate(entry_url):
        pdf_links = [entry_url]
        detail_links = []
    else:
        try:
            html = fetch_page(entry_url)
            pdf_links = extract_pdf_links(html, entry_url)
            detail_links = extract_detail_links(html, entry_url)
        except Exception as e:
            download_rows.append({
                "source_id": source_id,
                "seed_name": seed_name,
                "country_code": country_code,
                "agency_code": agency_code,
                "entry_url": entry_url,
                "detail_url": "",
                "pdf_url": "",
                "local_path": "",
                "file_size": 0,
                "status": f"entry_error: {e}",
            })
            continue

        for detail_url in detail_links:
            try:
                detail_html = fetch_page(detail_url)
                detail_pdf_links = extract_pdf_links(detail_html, detail_url)
                for pdf_url in detail_pdf_links:
                    detail_rows.append((detail_url, pdf_url))
            except Exception as e:
                download_rows.append({
                    "source_id": source_id,
                    "seed_name": seed_name,
                    "country_code": country_code,
                    "agency_code": agency_code,
                    "entry_url": entry_url,
                    "detail_url": detail_url,
                    "pdf_url": "",
                    "local_path": "",
                    "file_size": 0,
                    "status": f"detail_error: {e}",
                })

        for detail_url, pdf_url in detail_rows:
            pdf_links.append(pdf_url)

    if not pdf_links:
        download_rows.append({
            "source_id": source_id,
            "seed_name": seed_name,
            "country_code": country_code,
            "agency_code": agency_code,
            "entry_url": entry_url,
            "detail_url": "",
            "pdf_url": "",
            "local_path": "",
            "file_size": 0,
            "status": "no_pdf_found",
        })
        continue

    seen_urls = set()

    for idx, pdf_url in enumerate(pdf_links, start=1):
        if pdf_url in seen_urls or pdf_url in global_seen_urls:
            continue
        seen_urls.add(pdf_url)
        global_seen_urls.add(pdf_url)
        detail_url = ""
        for current_detail_url, current_pdf_url in detail_rows:
            if current_pdf_url == pdf_url:
                detail_url = current_detail_url
                break

        file_name = normalize_file_name(pdf_url, f"{country_code}_{agency_code}_{idx:04d}.pdf")
        if not is_practical_pdf(country_code, agency_code, file_name):
            download_rows.append({
                "source_id": source_id,
                "seed_name": seed_name,
                "country_code": country_code,
                "agency_code": agency_code,
                "entry_url": entry_url,
                "detail_url": detail_url,
                "pdf_url": pdf_url,
                "local_path": "",
                "file_size": 0,
                "status": "filtered_out",
            })
            continue

        save_path = save_dir / file_name

        try:
            file_size = download_pdf(pdf_url, save_path)
            status = "success"
        except Exception as e:
            file_size = 0
            status = f"download_error: {e}"

        download_rows.append({
            "source_id": source_id,
            "seed_name": seed_name,
            "country_code": country_code,
            "agency_code": agency_code,
            "entry_url": entry_url,
            "detail_url": detail_url,
            "pdf_url": pdf_url,
            "local_path": str(save_path) if status == "success" else "",
            "file_size": file_size,
            "status": status,
        })
        time.sleep(0.2)

download_df = pd.DataFrame(download_rows)
download_df.head()

## 4. 수집 결과를 확인한다

국가별, 기관별 성공 수를 먼저 보고, 빈 파일이나 중복 URL이 있는지 확인한다.


In [ ]:
download_df["status"].value_counts(dropna=False)

In [ ]:
download_df.groupby(["country_code", "agency_code"])["status"].count()

In [ ]:
download_df.duplicated(subset=["pdf_url"]).sum()

In [ ]:
download_df[download_df["file_size"] == 0].head()